In [10]:
##### Calculate the dissimilarity index and AIA for each sub-national region for each fold of spatial CV

import os
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
import rasterio
import rioxarray as rio
import matplotlib.colors as mcolors
from matplotlib.patches import Rectangle
import numpy as np
import pandas as pd
from scipy.spatial.distance import cdist

In [11]:
##### Load data

# Get the current working directory
cd = os.path.dirname(os.getcwd())

# Import predictors data
capital_model = pd.read_csv(f"{cd}/Data/Clean/Training_data/capital_relative_final_thinned.csv")
labor_model = pd.read_csv(f"{cd}/Data/Clean/Training_data/labor_relative_final_thinned.csv")

# spatial folds 
capital_folds = pd.read_csv(f"{cd}/Data/Fold_assignments/capital_folds.csv")
labor_folds = pd.read_csv(f"{cd}/Data/Fold_assignments/labor_folds.csv")

# Set file path to figure repo
fd = "/Users/carinamanitius/Library/CloudStorage/OneDrive-UniversityofVermont/Documents/OneDrive/Dissertation/Chapter 1/Figures/RESULTS/dissimilarity_spatial_CV"

In [12]:
#### Columns used as predictors

capital_cols = ['rtv_log_average_travel_time_port',
       'rtv_log_crop_intensity',
       'rtv_log_USD_production_per_million_HA',
       'rtv_log_tonnes_production_per_million_HA',
       'rtv_log_pop_density_people_per_100_km2',
       'rtv_log_cattle_density_per_100_km2',
       'rtv_log_sheep_density_per_100_km2',
       'rtv_log_livestock_density_LU_per_100_km2',
       'rtv_log_cereals_share_base_100_production_tonnes',
       'rtv_log_fruits_share_base_100_production_tonnes',
       'rtv_log_roots_tubers_share_base_100_production_tonnes',
       'rtv_log_vegetables_share_base_100_production_tonnes',
       'rtv_log_ruminants_share_base_100_production_tonnes',
       'rtv_log_share_base_100_large_field',
       'rtv_log_share_base_100_with_nightlights']

labor_cols = ['rtv_log_average_travel_time_port',
       'rtv_log_crop_intensity',
       'rtv_log_pop_density_people_per_100_km2',
       'rtv_log_cattle_density_per_100_km2',
       'rtv_log_livestock_density_LU_per_100_km2',
       'rtv_log_fruits_share_base_100_production_tonnes',
       'rtv_log_roots_tubers_share_base_100_production_tonnes',
       'rtv_log_rest_of_crops_share_base_100_production_tonnes',
       'rtv_log_sugar_crops_share_base_100_production_tonnes',
       'rtv_log_ruminants_share_base_100_production_tonnes',
       'rtv_log_pct_base_100_GDP_ag', 'rtv_log_share_base_100_large_field',
       'rtv_log_pct_base_100_cropland_irrigated']

In [13]:
##### Clean data

# isolate columns
capital_model = capital_model[capital_cols + ['country_ID']]
labor_model = labor_model[labor_cols + ['country_ID']]

# join with folds
capital_model = capital_model.merge(capital_folds, on='country_ID', how='left')
labor_model = labor_model.merge(labor_folds, on='country_ID', how='left')


In [14]:
###### Calculate average DI and AOA % for test set in each fold (CAPITAL)

results = []

for i in range(1, 6):
    fold_col = f"fold_{i}"

    train = capital_model.loc[capital_model[fold_col] == 0, capital_cols].copy()
    test  = capital_model.loc[capital_model[fold_col] == 1, capital_cols].copy()

    # --- 1. standardize using TRAIN mean/sd ---
    train_mean = train.mean()
    train_sd   = train.std(ddof=0).replace(0, 1)  

    train_s = (train - train_mean) / train_sd
    test_s  = (test  - train_mean) / train_sd

    train_arr = train_s.values
    test_arr  = test_s.values

    # --- 2. normalization factor: mean nearest-neighbor dist within train ---
    train_dist = cdist(train_arr, train_arr, metric="euclidean")
    np.fill_diagonal(train_dist, np.inf)          # exclude self-match
    train_nn_dist = train_dist.min(axis=1)
    norm_factor = train_nn_dist.mean()

    # --- 3. training DI's -> AOA threshold ---
    train_DI = train_nn_dist / norm_factor
    q75, q25 = np.percentile(train_DI, [75, 25])
    iqr = q75 - q25
    threshold = q75 + 1.5 * iqr

    # --- 4. DI for test points (distance to nearest train point) ---
    test_train_dist = cdist(test_arr, train_arr, metric="euclidean")
    test_nn_dist = test_train_dist.min(axis=1)
    test_DI = test_nn_dist / norm_factor

    # --- 5. AOA flag ---
    outside_aoa = test_DI > threshold
    pct_outside = 100 * outside_aoa.mean()

    results.append({
        "fold": i,
        "n_train": len(train),
        "n_test": len(test),
        "avg_DI_test": test_DI.mean(),
        "AOA_threshold": threshold,
        "pct_test_outside_AOA": pct_outside,
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

 fold  n_train  n_test  avg_DI_test  AOA_threshold  pct_test_outside_AOA
    1     1891     852     1.639036       2.036928             13.028169
    2     1690    1053     1.464339       1.940503             14.909782
    3     2443     300     1.557915       1.908461             23.666667
    4     2474     269     1.492476       1.927733             25.278810
    5     2474     269     1.305797       1.967564              3.717472


In [15]:
###### Calculate average DI and AOA % for test set in each fold (CAPITAL)

results = []

for i in range(1, 6):
    fold_col = f"fold_{i}"

    train = labor_model.loc[labor_model[fold_col] == 0, labor_cols].copy()
    test  = labor_model.loc[labor_model[fold_col] == 1, labor_cols].copy()

    # --- 1. standardize using TRAIN mean/sd ---
    train_mean = train.mean()
    train_sd   = train.std(ddof=0).replace(0, 1)  

    train_s = (train - train_mean) / train_sd
    test_s  = (test  - train_mean) / train_sd

    train_arr = train_s.values
    test_arr  = test_s.values

    # --- 2. normalization factor: mean nearest-neighbor dist within train ---
    train_dist = cdist(train_arr, train_arr, metric="euclidean")
    np.fill_diagonal(train_dist, np.inf)          # exclude self-match
    train_nn_dist = train_dist.min(axis=1)
    norm_factor = train_nn_dist.mean()

    # --- 3. training DI's -> AOA threshold ---
    train_DI = train_nn_dist / norm_factor
    q75, q25 = np.percentile(train_DI, [75, 25])
    iqr = q75 - q25
    threshold = q75 + 1.5 * iqr

    # --- 4. DI for test points (distance to nearest train point) ---
    test_train_dist = cdist(test_arr, train_arr, metric="euclidean")
    test_nn_dist = test_train_dist.min(axis=1)
    test_DI = test_nn_dist / norm_factor

    # --- 5. AOA flag ---
    outside_aoa = test_DI > threshold
    pct_outside = 100 * outside_aoa.mean()

    results.append({
        "fold": i,
        "n_train": len(train),
        "n_test": len(test),
        "avg_DI_test": test_DI.mean(),
        "AOA_threshold": threshold,
        "pct_test_outside_AOA": pct_outside,
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

 fold  n_train  n_test  avg_DI_test  AOA_threshold  pct_test_outside_AOA
    1     2698     868     1.738824       2.013572             26.497696
    2     2519    1047     1.430323       2.005101              6.017192
    3     3017     549     1.433569       2.073883              7.103825
    4     3010     556     1.495829       2.027501             14.568345
    5     3020     546     1.450606       2.015513             12.637363
